# Step 02: Data Preparation, Target Centering & Feature Extraction

Executes spatial bounds filtering, SNR noise filtering, target-relative median centering (\Delta x = x - \text{median}(x), \Delta y = y - \text{median}(y)), and uniform $N=32$ point resampling to produce preprocessed Train, Validation, and Test feature tensors.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root_dir) not in sys.path: sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset_parser import get_loso_splits, load_recording_df, extract_label
from src.transforms import extract_raw_windows_from_df, FoldScaler, transform_representation

sns.set_theme(style="whitegrid")


In [ ]:
# Execute Full Preprocessing & Target-Relative Centering Pipeline for Fold 0
splits = get_loso_splits()
fold0 = splits[0]

def extract_windows_for_file_list(file_list):
    wins, lsl = [], []
    for fpath in file_list:
        lbl = extract_label(fpath)
        df_clean = load_recording_df(fpath)
        w_arrs = extract_raw_windows_from_df(df_clean, window_size_frames=10, stride=5)
        for w in w_arrs:
            wins.append(w)
            lsl.append(lbl)
    return np.array(wins, dtype=np.float32), np.array(lsl, dtype=np.int64)

X_tr_raw, y_tr = extract_windows_for_file_list(fold0["train_files"])
X_va_raw, y_va = extract_windows_for_file_list(fold0["val_files"])
X_te_raw, y_te = extract_windows_for_file_list(fold0["test_files"])

# Apply leakage-free feature scaling fitted strictly on training fold
scaler = FoldScaler()
scaler.fit(X_tr_raw)
X_tr_scaled = scaler.transform(X_tr_raw)
X_va_scaled = scaler.transform(X_va_raw)
X_te_scaled = scaler.transform(X_te_raw)

print(f"Preprocessed Train Tensors: {X_tr_scaled.shape} (Falls: {sum(y_tr==1)}, ADLs: {sum(y_tr==0)})")
print(f"Preprocessed Val Tensors:   {X_va_scaled.shape} (Falls: {sum(y_va==1)}, ADLs: {sum(y_va==0)})")
print(f"Preprocessed Test Tensors:  {X_te_scaled.shape} (Falls: {sum(y_te==1)}, ADLs: {sum(y_te==0)})")


In [ ]:
# Preview Preprocessed Centered Point Cloud Tensor
sample_clip = X_tr_scaled[0, 5]  # Clip 0, Frame 5
fig = plt.figure(figsize=(8, 5), dpi=100)
ax = fig.add_subplot(1, 1, 1, projection='3d')
sc = ax.scatter(sample_clip[:, 0], sample_clip[:, 1], sample_clip[:, 2], c=sample_clip[:, 3], cmap='coolwarm', s=60, edgecolors='black')
ax.set_title("Preprocessed Centered Point Cloud (Frame 5)\n[Delta X, Delta Y, Z, v] Standardized N=32 Points", fontweight='bold')
ax.set_xlabel("Delta X (m)"); ax.set_ylabel("Delta Y (m)"); ax.set_zlabel("Elevation Z (m)")
plt.colorbar(sc, ax=ax, shrink=0.6, label='Doppler Velocity (m/s)')
plt.tight_layout()
plt.show()
